In [1]:
# Get ICD10, ICD9, and procedural codes for full srWGS cohort. Get data on statin use (Rx history and surveys)
# Get LDL and trig levels
# Get demographics, BMI, and date of last ICD code

# Setup

In [2]:
library(tidyverse)
library(bigrquery)

In [3]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_99802609_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

In [4]:
# Check that the necessary environment variables are present
message(Sys.getenv("WORKSPACE_CDR"))
message(Sys.getenv("GOOGLE_PROJECT"))

In [108]:
# Constants (mostly paths to resources)
analysis_bucket <- 'gs://cvfg-analyses-bucket-wb-spotless-pepper-7507'
intermediate_tables_dir <- file.path(
    analysis_bucket,
    'cvfg-analyses_2026-09-01',
    'intermediates'
)
intermediate_tables_dir_local <- file.path(
    '~',
    'workspace',
    'cvfg-analyses-bucket',
    'cvfg-analyses_2026-09-01',
    'intermediates'
)

# Get MI and ischemic heart disease codes -DONE

## ICD codes

### Do once (done)

In [8]:
# This query represents dataset "MI and ICHD ICD codes" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_99802609_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (
                                    1326588, 1569127, 1569134, 1569135, 1569145, 35207684, 35207685, 35207686, 35207687, 35207688, 35207689, 35207691, 35207692, 35207693, 35207695, 35207696, 35207697, 35207698, 35207699, 35207700, 35207701, 35207702, 35207704, 35207705, 35207706, 44819697, 44819699, 44819700, 44819702, 44820857, 44820858, 44820859, 44820860, 44820861, 44820862, 44820863, 44820864, 44823111, 44823120, 44824237, 44825428, 44825429, 44825430, 44826635, 44826636, 44826643, 44827782, 44827783, 44828972, 44828973, 44830079, 44830080, 44831236, 44831237, 44831238, 44832372, 44832373, 44832374, 44832375, 44832376, 44833561, 44834718, 44834719, 44834720, 44834721, 44834723, 44834724, 44834725, 44834733, 44835926, 44835927, 44835928, 44835929, 44835932, 44837099, 45533436, 45548013, 45557536, 45562340, 45562344, 45567167, 45567168, 45572079, 45572080, 45576865, 45576866, 45586572, 45591456, 45596197, 45596199, 45601024, 45605779, 45605781, 45605787, 45605788
                                )       
                                AND full_text LIKE '%_rank1]%'      
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1
                        )
                )  
                AND (
                    c_occurrence.PERSON_ID IN (
                        SELECT
                            distinct person_id  
                        FROM
                            `cb_search_person` cb_search_person  
                        WHERE
                            cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 
                            ) 
                        )
                )
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_99802609_path <- file.path(
  intermediate_tables_dir,
  "condition_99802609",
  "condition_99802609_*.csv")
message(str_glue('The data will be written to {condition_99802609_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_99802609_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_99802609_path,
  destination_format = "CSV")




### Format ICD code data

In [17]:
# Set if data pull was run in a different session
condition_99802609_path <- file.path(
  intermediate_tables_dir,
  "condition_99802609",
  "condition_99802609_*.csv")
#condition_99802609_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240321/condition_99802609/condition_99802609_*.csv'

In [18]:
icd_df <- read_bq_export_from_workspace_bucket(condition_99802609_path)
dim(icd_df)
head(icd_df, 5)

In [19]:
icd_df %>% group_by(source_concept_name,source_concept_code) %>% summarize(n=n())

In [20]:
#some entries may be child codes and will need to be mapped to single UKB field id ...?
#example: ICD parent code Z85 maps to UKB field id 41270_Z850
#Z85 child codes include Z85.040, Z85.05, etc.

In [21]:
#for now, just need to know if a subject has any codes and date of first code

In [22]:
#keep just the earliest code for each subject
icd_df$condition_start_datetime <- as.Date(icd_df$condition_start_datetime)
icd_df <- icd_df %>% group_by(person_id) %>% filter(condition_start_datetime == min(condition_start_datetime))
dim(icd_df)

In [23]:
#some subjects still have multiple measurements. Keep just one reading per subject
icd_df <- distinct(icd_df,person_id,.keep_all=T)
dim(icd_df)

In [24]:
icd_df <- icd_df %>% select(person_id,source_concept_name,source_concept_code,condition_start_datetime)
head(icd_df)
dim(icd_df)

## Procedural codes

### Do once (done)

In [8]:
# This query represents dataset "Coronary athero-related CPT codes" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v7
dataset_35162265_procedure_sql <- paste("
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `procedure_occurrence` procedure 
        WHERE
            (
                procedure_source_concept_id IN (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (
                                    2107216, 2107217, 2107218, 2107219, 2107220, 2107221, 2107222, 2107223, 2107224, 2107226, 2107227, 2107228, 2107231, 2107242, 2107243, 2107244, 2107250, 2313796, 2313801, 2313802, 2313803, 2313804, 2313810, 2313811, 43527908, 43527909, 43527994, 43527995, 43527996, 43527997, 43527998, 43527999, 43528000, 43528001, 43528002, 43528003, 43528004
                                )       
                                AND full_text LIKE '%_rank1]%'      
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1
                        )
                )  
                AND (
                    procedure.PERSON_ID IN (
                        SELECT
                            distinct person_id  
                        FROM
                            `cb_search_person` cb_search_person  
                        WHERE
                            cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 
                            ) 
                        )
                )
            ) procedure 
        LEFT JOIN
            `concept` p_standard_concept 
                ON procedure.procedure_concept_id = p_standard_concept.concept_id 
        LEFT JOIN
            `concept` p_type 
                ON procedure.procedure_type_concept_id = p_type.concept_id 
        LEFT JOIN
            `concept` p_modifier 
                ON procedure.modifier_concept_id = p_modifier.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON procedure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` p_visit 
                ON v.visit_concept_id = p_visit.concept_id 
        LEFT JOIN
            `concept` p_source_concept 
                ON procedure.procedure_source_concept_id = p_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedure_35162265_path <- file.path(
  intermediate_tables_dir,
  "procedure_35162265",
  "procedure_35162265_*.csv")
message(str_glue('The data will be written to {procedure_35162265_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_35162265_procedure_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  procedure_35162265_path,
  destination_format = "CSV")

### Format procedure code data

In [9]:
#procedure_35162265_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240321/procedure_35162265/procedure_35162265_*.csv'
rocedure_35162265_path <- file.path(
  intermediate_tables_dir,
  "procedure_35162265",
  "procedure_35162265_*.csv")

In [10]:
CPT_df <- read_bq_export_from_workspace_bucket(procedure_35162265_path)

dim(CPT_df)

head(CPT_df, 5)

In [11]:
CPT_df %>% group_by(source_concept_name,source_concept_code) %>% summarize(n=n())

In [12]:
#for now, just need to know if a subject has any codes and date of first code
#keep just the earliest code for each subject
CPT_df$procedure_datetime <- as.Date(CPT_df$procedure_datetime)
CPT_df <- CPT_df %>% group_by(person_id) %>% filter(procedure_datetime == min(procedure_datetime))
dim(CPT_df)

In [13]:
#some subjects still have multiple measurements. Keep just one code per subject
CPT_df <- distinct(CPT_df,person_id,.keep_all=T)
dim(CPT_df)

In [14]:
CPT_df <- CPT_df %>% select(person_id,source_concept_name,source_concept_code,procedure_datetime)
head(CPT_df)
dim(CPT_df)

## Join condition and procedure codes

In [25]:
names(icd_df)

In [26]:
colnames(icd_df) <- c('person_id','source_concept_name','source_concept_code','datetime')
head(icd_df)

In [27]:
colnames(CPT_df) <- c('person_id','source_concept_name','source_concept_code','datetime')
head(CPT_df)

In [28]:
codes_df <- rbind(icd_df,CPT_df)
dim(codes_df)
head(codes_df)

In [29]:
#keep just the earliest code for each subject
codes_df <- codes_df %>% group_by(person_id) %>% filter(datetime == min(datetime))
dim(codes_df)

In [30]:
#some subjects still have multiple measurements. Keep just one code per subject
codes_df <- distinct(codes_df,person_id,.keep_all=T)
dim(codes_df)

In [31]:
colnames(codes_df) <- c('person_id','CAD_code_name','CAD_code','CAD_code_date'
                       )
head(codes_df)

# Get PAD codes - DONE

In [32]:
# This query represents dataset "Temp PAD" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_32633938_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (317309, 3654996)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_32633938_path <- file.path(
  intermediate_tables_dir,
  "condition_32633938",
  "condition_32633938_*.csv")
message(str_glue('The data will be written to {condition_32633938_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_32633938_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_32633938_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_32633938_path}` to copy these files
#       to the Jupyter disk.

PAD_df <- read_bq_export_from_workspace_bucket(condition_32633938_path)

dim(PAD_df)

head(PAD_df, 5)

In [35]:
PAD_path <- file.path(
  intermediate_tables_dir,
  "condition_32633938",
  "condition_32633938_*.csv")
#PAD_path = 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20241101/condition_32633938/condition_32633938_000000000000.csv'

In [36]:
PAD_df <- read_bq_export_from_workspace_bucket(PAD_path)

dim(PAD_df)

head(PAD_df, 5)

In [37]:
PAD_df %>% group_by(standard_concept_name) %>% summarize(n=n())

In [38]:
nrow(distinct(PAD_df,person_id))

In [39]:
names(PAD_df)

In [40]:
PAD_df <- PAD_df[,c('person_id','standard_concept_name','standard_concept_code','condition_start_datetime')]
head(PAD_df)

In [41]:
PAD_df$condition_start_datetime <- as.Date(PAD_df$condition_start_datetime)

In [42]:
#keep just the earliest code for each subject
PAD_df <- PAD_df %>% group_by(person_id) %>% filter(condition_start_datetime == min(condition_start_datetime))
dim(PAD_df)

In [43]:
#some subjects still have multiple measurements. Keep just one code per subject
PAD_df <- distinct(PAD_df,person_id,.keep_all=T)
dim(PAD_df)

In [44]:
colnames(PAD_df) <- c('person_id','PAD_code_name','PAD_code','PAD_code_date'
                       )

# Get FH codes - DONE

In [45]:
# This query represents dataset "Temp FH" for domain "observation" and was generated for All of Us Controlled Tier Dataset v7
dataset_64642663_observation_sql <- paste("
    SELECT
        observation.person_id,
        observation.observation_concept_id,
        o_standard_concept.concept_name as standard_concept_name,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        observation.observation_datetime,
        observation.observation_type_concept_id,
        o_type.concept_name as observation_type_concept_name,
        observation.value_as_number,
        observation.value_as_string,
        observation.value_as_concept_id,
        o_value.concept_name as value_as_concept_name,
        observation.qualifier_concept_id,
        o_qualifier.concept_name as qualifier_concept_name,
        observation.unit_concept_id,
        o_unit.concept_name as unit_concept_name,
        observation.visit_occurrence_id,
        o_visit.concept_name as visit_occurrence_concept_name,
        observation.observation_source_value,
        observation.observation_source_concept_id,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `observation` observation 
        WHERE
            (
                observation_source_concept_id IN (37202301)
            )  
            AND (
                observation.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) observation 
    LEFT JOIN
        `concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `concept` o_type 
            ON observation.observation_type_concept_id = o_type.concept_id 
    LEFT JOIN
        `concept` o_value 
            ON observation.value_as_concept_id = o_value.concept_id 
    LEFT JOIN
        `concept` o_qualifier 
            ON observation.qualifier_concept_id = o_qualifier.concept_id 
    LEFT JOIN
        `concept` o_unit 
            ON observation.unit_concept_id = o_unit.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON observation.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` o_visit 
            ON v.visit_concept_id = o_visit.concept_id 
    LEFT JOIN
        `concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
observation_64642663_path <- file.path(
  intermediate_tables_dir,
  "observation_64642663",
  "observation_64642663_*.csv")
message(str_glue('The data will be written to {observation_64642663_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_64642663_observation_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  observation_64642663_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {observation_64642663_path}` to copy these files
#       to the Jupyter disk.
FH_df <- read_bq_export_from_workspace_bucket(observation_64642663_path)

dim(FH_df)

head(FH_df, 5)

In [ ]:
FH_PATH <- file.path(
  intermediate_tables_dir,
  "observation_64642663",
  "observation_64642663_*.csv")
#FH_path = 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20241101/observation_64642663/observation_64642663_000000000000.csv'

In [ ]:
FH_df <- read_bq_export_from_workspace_bucket(FH_path)

dim(FH_df)

head(FH_df, 5)

In [46]:
FH_df %>% group_by(standard_concept_name) %>% summarize(n=n())

In [47]:
FH_df <- FH_df %>% filter(standard_concept_name == 'Family history of familial hypercholesterolemia'
                         )

In [48]:
FH_df %>% group_by(standard_concept_name) %>% summarize(n=n())

In [49]:
FH_df <- FH_df[,c('person_id','standard_concept_name','standard_concept_code','observation_datetime')]
colnames(FH_df) <- c('person_id','FH_code_name','FH_code','FH_code_date'
                       )
head(FH_df)

In [50]:
nrow(distinct(FH_df,person_id))

In [52]:
# This query represents dataset "Temp FH" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_64642663_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (37200313)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_64642663_path <- file.path(
  intermediate_tables_dir,
  "condition_64642663",
  "condition_64642663_*.csv")
message(str_glue('The data will be written to {condition_64642663_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_64642663_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_64642663_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_64642663_path}` to copy these files
#       to the Jupyter disk.

FH_df2 <- read_bq_export_from_workspace_bucket(condition_64642663_path)

dim(FH_df2)

head(FH_df2, 5)

In [ ]:
FH_path2 <- file.path(
  intermediate_tables_dir,
  "condition_64642663",
  "condition_64642663_*.csv")

#FH_path2 = 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20241101/condition_64642663/condition_64642663_000000000000.csv'

In [ ]:
FH_df2 <- read_bq_export_from_workspace_bucket(FH_path2)

dim(FH_df2)

head(FH_df2, 5)

In [53]:
FH_df2 %>% group_by(standard_concept_name) %>% summarize(n=n())

In [54]:
FH_df2 <- FH_df2[,c('person_id','standard_concept_name','standard_concept_code','condition_start_datetime')]
colnames(FH_df2) <- c('person_id','FH_code_name','FH_code','FH_code_date'
                       )
head(FH_df2)

In [55]:
nrow(distinct(FH_df2,person_id))

In [56]:
FH_df <- rbind(FH_df,FH_df2)
dim(FH_df)
head(FH_df)

In [57]:
nrow(distinct(FH_df,person_id))

In [58]:
FH_df$FH_code_date <- as.Date(FH_df$FH_code_date)
#keep just the earliest code for each subject
FH_df <- FH_df %>% group_by(person_id) %>% filter(FH_code_date == min(FH_code_date))
dim(FH_df)

In [59]:
#some subjects still have multiple measurements. Keep just one code per subject
FH_df <- distinct(FH_df,person_id,.keep_all=T)
dim(FH_df)

# Get CAD family history -DONE

In [60]:

# This query represents dataset "Temp family history CAD" for domain "observation" and was generated for All of Us Controlled Tier Dataset v7
dataset_59227507_observation_sql <- paste("
    SELECT
        observation.person_id,
        observation.observation_concept_id,
        o_standard_concept.concept_name as standard_concept_name,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        observation.observation_datetime,
        observation.observation_type_concept_id,
        o_type.concept_name as observation_type_concept_name,
        observation.value_as_number,
        observation.value_as_string,
        observation.value_as_concept_id,
        o_value.concept_name as value_as_concept_name,
        observation.qualifier_concept_id,
        o_qualifier.concept_name as qualifier_concept_name,
        observation.unit_concept_id,
        o_unit.concept_name as unit_concept_name,
        observation.visit_occurrence_id,
        o_visit.concept_name as visit_occurrence_concept_name,
        observation.observation_source_value,
        observation.observation_source_concept_id,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `observation` observation 
        WHERE
            (
                observation_concept_id IN (4047566)
            )  
            AND (
                observation.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) observation 
    LEFT JOIN
        `concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `concept` o_type 
            ON observation.observation_type_concept_id = o_type.concept_id 
    LEFT JOIN
        `concept` o_value 
            ON observation.value_as_concept_id = o_value.concept_id 
    LEFT JOIN
        `concept` o_qualifier 
            ON observation.qualifier_concept_id = o_qualifier.concept_id 
    LEFT JOIN
        `concept` o_unit 
            ON observation.unit_concept_id = o_unit.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON observation.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` o_visit 
            ON v.visit_concept_id = o_visit.concept_id 
    LEFT JOIN
        `concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
observation_59227507_path <- file.path(
  intermediate_tables_dir,
  "observation_59227507",
  "observation_59227507_*.csv")
message(str_glue('The data will be written to {observation_59227507_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_59227507_observation_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  observation_59227507_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {observation_59227507_path}` to copy these files
#       to the Jupyter disk.
CADFH_df <- read_bq_export_from_workspace_bucket(observation_59227507_path)

dim(CADFH_df)

head(CADFH_df, 5)

In [ ]:
#CADFH_path = 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20241101/observation_59227507/observation_59227507_*.csv'

In [ ]:
CADFH_df <- read_bq_export_from_workspace_bucket(CADFH_path)

dim(CADFH_df)

head(CADFH_df, 5)

In [61]:
CADFH_df %>% group_by(standard_concept_name) %>% summarize(n=n())

In [62]:
CADFH_df <- CADFH_df [,c('person_id','standard_concept_name','standard_concept_code','observation_datetime')]
colnames(CADFH_df ) <- c('person_id','CADFH_code_name','CADFH_code','CADFH_code_date'
                       )
head(CADFH_df)

In [63]:
nrow(distinct(CADFH_df,person_id))

In [64]:
CADFH_df$CADFH_code_date <- as.Date(CADFH_df$CADFH_code_date)
#keep just the earliest code for each subject
CADFH_df <- CADFH_df %>% group_by(person_id) %>% filter(CADFH_code_date == min(CADFH_code_date))
dim(CADFH_df)

In [65]:
#some subjects still have multiple measurements. Keep just one code per subject
CADFH_df <- distinct(CADFH_df,person_id,.keep_all=T)
dim(CADFH_df)

# Get hypercholesterolemia codes

In [66]:
# This query represents dataset "Temp hypercholesterolemia" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_86884566_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (35207060, 37200312, 37200313, 44834564)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_86884566_path <- file.path(
  intermediate_tables_dir,  # Comment out this line if you want the export to always overwrite.
  "condition_86884566",
  "condition_86884566_*.csv")
message(str_glue('The data will be written to {condition_86884566_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_86884566_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_86884566_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_86884566_path}` to copy these files
#       to the Jupyter disk.

hc_df <- read_bq_export_from_workspace_bucket(condition_86884566_path)

dim(hc_df)

head(hc_df, 5)

In [ ]:
#hc_path = 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20241104/condition_86884566/condition_86884566_*.csv'

In [67]:
hc_df <- read_bq_export_from_workspace_bucket(hc_path)

dim(hc_df)

head(hc_df, 5)

In [68]:
hc_df %>% group_by(source_concept_name,source_concept_code) %>% summarize(n=n())

In [69]:
#keep just the earliest code for each subject
hc_df$condition_start_datetime <- as.Date(hc_df$condition_start_datetime)
hc_df <- hc_df %>% group_by(person_id) %>% filter(condition_start_datetime == min(condition_start_datetime))
dim(hc_df)

In [70]:
nrow(distinct(hc_df,person_id))

In [71]:
#some subjects still have multiple measurements. Keep just one reading per subject
hc_df <- distinct(hc_df,person_id,.keep_all=T)
dim(hc_df)

In [72]:
hc_df <- hc_df %>% select(person_id,source_concept_name,source_concept_code,condition_start_datetime)
head(hc_df)
dim(hc_df)

In [73]:
colnames(hc_df) <- c('person_id','HC_code_name','HC_code','HC_code_date'
                       )
head(hc_df)

In [74]:
hc_df <- hc_df %>% select('person_id','HC_code_date')

In [75]:
hc_df$HC_code = 1
head(hc_df)

# Get medication history - DONE

## Do once (done)

### Statin use

In [76]:
# This query represents dataset "Statins" for domain "drug" and was generated for All of Us Controlled Tier Dataset v7
dataset_32584860_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (
                    SELECT
                        DISTINCT ca.descendant_id 
                    FROM
                        `cb_criteria_ancestor` ca 
                    JOIN
                        (
                            SELECT
                                DISTINCT c.concept_id       
                            FROM
                                `cb_criteria` c       
                            JOIN
                                (
                                    SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (
                                            1510813, 1539403, 1545958, 1549686, 1551860, 1592085, 1592180, 40165636
                                        )             
                                        AND full_text LIKE '%_rank1]%'       
                                ) a 
                                    ON (
                                        c.path LIKE CONCAT('%.',
                                    a.id,
                                    '.%') 
                                    OR c.path LIKE CONCAT('%.',
                                    a.id) 
                                    OR c.path LIKE CONCAT(a.id,
                                    '.%') 
                                    OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1
                                ) b 
                                    ON (
                                        ca.ancestor_id = b.concept_id
                                    )
                            )
                        )  
                        AND (
                            d_exposure.PERSON_ID IN (
                                SELECT
                                    distinct person_id  
                            FROM
                                `cb_search_person` cb_search_person  
                            WHERE
                                cb_search_person.person_id IN (
                                    SELECT
                                        person_id 
                                    FROM
                                        `cb_search_person` p 
                                    WHERE
                                        has_whole_genome_variant = 1 
                                ) 
                            )
                    )
                ) d_exposure 
            LEFT JOIN
                `concept` d_standard_concept 
                    ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
            LEFT JOIN
                `concept` d_type 
                    ON d_exposure.drug_type_concept_id = d_type.concept_id 
            LEFT JOIN
                `concept` d_route 
                    ON d_exposure.route_concept_id = d_route.concept_id 
            LEFT JOIN
                `visit_occurrence` v 
                    ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
            LEFT JOIN
                `concept` d_visit 
                    ON v.visit_concept_id = d_visit.concept_id 
            LEFT JOIN
                `concept` d_source_concept 
                    ON d_exposure.drug_source_concept_id = d_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_32584860_path <- file.path(
  intermediate_tables_dir,
  "drug_32584860",
  "drug_32584860_*.csv")
message(str_glue('The data will be written to {drug_32584860_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_32584860_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_32584860_path,
  destination_format = "CSV")


### Non-statin cholesterol medications

In [77]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "temp2" for domain "drug" and was generated for All of Us Controlled Tier Dataset v7
dataset_41010260_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (
                    SELECT
                        DISTINCT ca.descendant_id 
                    FROM
                        `cb_criteria_ancestor` ca 
                    JOIN
                        (
                            SELECT
                                DISTINCT c.concept_id       
                            FROM
                                `cb_criteria` c       
                            JOIN
                                (
                                    SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `cb_criteria` cr             
                                    WHERE
                                        concept_id IN (
                                            1501617, 1517824, 1518148, 1526475, 1551803, 1558242, 19095309, 37499009, 43560137, 46275447, 46287466
                                        )             
                                        AND full_text LIKE '%_rank1]%'       
                                ) a 
                                    ON (
                                        c.path LIKE CONCAT('%.',
                                    a.id,
                                    '.%') 
                                    OR c.path LIKE CONCAT('%.',
                                    a.id) 
                                    OR c.path LIKE CONCAT(a.id,
                                    '.%') 
                                    OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1
                                ) b 
                                    ON (
                                        ca.ancestor_id = b.concept_id
                                    )
                            )
                        )  
                        AND (
                            d_exposure.PERSON_ID IN (
                                SELECT
                                    distinct person_id  
                            FROM
                                `cb_search_person` cb_search_person  
                            WHERE
                                cb_search_person.person_id IN (
                                    SELECT
                                        person_id 
                                    FROM
                                        `cb_search_person` p 
                                    WHERE
                                        has_whole_genome_variant = 1 
                                ) 
                            )
                    )
                ) d_exposure 
            LEFT JOIN
                `concept` d_standard_concept 
                    ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
            LEFT JOIN
                `concept` d_type 
                    ON d_exposure.drug_type_concept_id = d_type.concept_id 
            LEFT JOIN
                `concept` d_route 
                    ON d_exposure.route_concept_id = d_route.concept_id 
            LEFT JOIN
                `visit_occurrence` v 
                    ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
            LEFT JOIN
                `concept` d_visit 
                    ON v.visit_concept_id = d_visit.concept_id 
            LEFT JOIN
                `concept` d_source_concept 
                    ON d_exposure.drug_source_concept_id = d_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_41010260_path <- file.path(
  intermediate_tables_dir,
  "drug_41010260",
  "drug_41010260_*.csv")
message(str_glue('The data will be written to {drug_41010260_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_41010260_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_41010260_path,
  destination_format = "CSV")

### Survey response

In [78]:
# This query represents dataset "temp" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_43208585_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (
                43528793
            )
        )  
        AND (
            answer.PERSON_ID IN (
                SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (
                        SELECT
                            person_id 
                        FROM
                            `cb_search_person` p 
                        WHERE
                            has_whole_genome_variant = 1 
                    ) 
                )
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_43208585_path <- file.path(
  intermediate_tables_dir,
  "survey_43208585",
  "survey_43208585_*.csv")
message(str_glue('The data will be written to {survey_43208585_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_43208585_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_43208585_path,
  destination_format = "CSV")

## Format medication data

### Survey data

In [ ]:
#survey_43208585_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240321/survey_43208585/survey_43208585_*.csv'

In [79]:
survey_df <- read_bq_export_from_workspace_bucket(survey_43208585_path)

dim(survey_df)

head(survey_df, 5)

In [80]:
survey_df %>% group_by(answer) %>% summarize(n=n())

In [81]:
survey_df %>% group_by(question) %>% summarize(n=n())

In [82]:
survey_df <- select(survey_df, person_id, survey_datetime, answer)
head(survey_df)

In [83]:
temp <- survey_df %>% mutate(ans = case_when(
                                   answer == "Are you currently prescribed medications and/or receiving treatment for high cholesterol? - Yes" ~ "Yes",
                                   .default = "No" 
                                ) 
                            )

temp %>% group_by(answer,ans) %>% summarize(n=n())

In [84]:
survey_df <- temp
head(survey_df)

In [85]:
survey_df <- select(survey_df,person_id, survey_datetime,ans)
head(survey_df)

In [86]:
survey_df$survey_datetime <- as.Date(survey_df$survey_datetime)

### Statins

In [ ]:
#drug_32584860_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240321/drug_32584860/drug_32584860_*.csv'

In [87]:
statin_df <- read_bq_export_from_workspace_bucket(drug_32584860_path)

dim(statin_df)
head(statin_df, 5)

In [88]:
statin_df %>% group_by(standard_concept_name) %>% summarize(n=n())

In [89]:
#for each person, get earliest date of statin Rx
statin_df$drug_exposure_start_datetime<-as.Date(statin_df$drug_exposure_start_datetime)
statin_df <- statin_df %>% group_by(person_id) %>% filter(drug_exposure_start_datetime == min(drug_exposure_start_datetime))

In [90]:
dim(statin_df)
dim(distinct(statin_df,person_id))

In [91]:
#Keep just one row per subject
statin_df <- distinct(statin_df,person_id,.keep_all=T)
head(statin_df)

In [92]:
statin_df <- select(statin_df,person_id,drug_exposure_start_datetime)
statin_df$on_statin <- 'yes'
head(statin_df)

In [93]:
colnames(statin_df)<-c('person_id','statin_start_date','on_statin')
statin_df <- statin_df %>% relocate(statin_start_date, .after = on_statin)
head(statin_df)

### Non-statin drugs

In [ ]:
#drug_41010260_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240321/drug_41010260/drug_41010260_*.csv'

In [94]:
nonstatin_df <- read_bq_export_from_workspace_bucket(drug_41010260_path)

dim(nonstatin_df)

head(nonstatin_df, 5)

In [95]:
nonstatin_df %>% group_by(standard_concept_name) %>% summarize(n=n())

In [96]:
#ok to keep combo meds with statin and non-statin? I think so, since these should also be in the statin set, and we don't
#really care about examining non-statins by themselves, but just in addition to the statin category

In [97]:
#for each person, get earliest date of nonstatin Rx
nonstatin_df$drug_exposure_start_datetime<-as.Date(nonstatin_df$drug_exposure_start_datetime)
nonstatin_df <- nonstatin_df %>% group_by(person_id) %>% filter(drug_exposure_start_datetime == min(drug_exposure_start_datetime))

In [98]:
dim(nonstatin_df)
dim(distinct(nonstatin_df,person_id))

In [99]:
#Keep just one row per subject
nonstatin_df <- distinct(nonstatin_df,person_id,.keep_all=T)
head(nonstatin_df)

In [100]:
nonstatin_df <- select(nonstatin_df,person_id,drug_exposure_start_datetime)
nonstatin_df$on_nonstatin <- 'yes'
head(nonstatin_df)

In [101]:
colnames(nonstatin_df)<-c('person_id','nonstatin_start_date','on_nonstatin')
nonstatin_df <- nonstatin_df %>% relocate(nonstatin_start_date, .after = on_nonstatin)
head(nonstatin_df)

### Join medication data

In [102]:
meds_df <- full_join(statin_df,nonstatin_df,by='person_id')
head(meds_df)
dim(meds_df)

In [103]:
meds_df <- full_join(meds_df,survey_df,by='person_id')
head(meds_df)
dim(meds_df)

In [104]:
dim(distinct(meds_df,person_id))

In [105]:
meds_df <- meds_df %>% mutate(any_chol_med = case_when(
                              on_statin == 'yes' | on_nonstatin == 'yes' | ans == 'Yes' ~ 'yes',  
                              .default = 'no'
                                ))

meds_df %>% group_by(any_chol_med,on_statin,on_nonstatin,ans) %>% summarize(n=n())

In [106]:
head(meds_df)

In [109]:
# store the dataframe in current workspace
write_excel_csv(
    meds_df,
    file.path(intermediate_tables_dir_local, 'LDLR_medications_intermediate_datasave.csv')
)

In [ ]:
#now get date of earliest med
#ugh
#if any_chol_med yes, then for each of on_statin, on_nonstatin, and ans, if answer is yes, compare dates and 
#keep earliest

#pseudo code: 
#if any_chol_med == yes, then
    #if on_statin == yes & on_nonstatin == yes & ans == yes
    #then date = min(statin_start_date,nonstatin_start_date,survey datetime)
    #else ....
    
#there has to be a better way to do this

In [110]:
#test whether any dates are associated with negative entries
meds_df[meds_df$on_statin=='no',] %>% group_by(statin_start_date) %>% summarize(n=n())

In [111]:
meds_df[meds_df$on_nonstatin=='no',] %>% group_by(nonstatin_start_date) %>% summarize(n=n())

In [112]:
meds_df[meds_df$ans=='NA',] %>% group_by(survey_datetime) %>% summarize(n=n())

In [113]:
meds_df[meds_df$ans=='No',] %>% group_by(survey_datetime) %>% summarize(n=n())

In [114]:
meds_df$survey_datetime[meds_df$ans=='No'] <- NA
meds_df[meds_df$ans=='No',] %>% group_by(survey_datetime) %>% summarize(n=n())

In [115]:
head(meds_df)

In [116]:
meds_df <- meds_df %>% rowwise() %>% mutate(any_chol_med_start_date = min(statin_start_date,nonstatin_start_date,survey_datetime,
                                                           na.rm=TRUE))

In [117]:
head(meds_df)

In [118]:
meds_df$any_chol_med_start_date[meds_df$any_chol_med=='no'] <- NA

In [119]:
dim(distinct(meds_df,person_id))

In [120]:
dim(meds_df)

In [121]:
meds_df <- select(meds_df,person_id,any_chol_med,any_chol_med_start_date)
head(meds_df)

# Get lab data -DONE

## Do once (done)

In [122]:
# This query represents dataset "Temp ldl trig" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v7
dataset_65837970_measurement_sql <- paste("
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (
                                    3022192, 37026687
                                )       
                                AND full_text LIKE '%_rank1]%'      
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )  
                AND (
                    measurement.PERSON_ID IN (
                        SELECT
                            distinct person_id  
                        FROM
                            `cb_search_person` cb_search_person  
                        WHERE
                            cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 
                            ) 
                        )
                )
            ) measurement 
        LEFT JOIN
            `concept` m_standard_concept 
                ON measurement.measurement_concept_id = m_standard_concept.concept_id 
        LEFT JOIN
            `concept` m_type 
                ON measurement.measurement_type_concept_id = m_type.concept_id 
        LEFT JOIN
            `concept` m_operator 
                ON measurement.operator_concept_id = m_operator.concept_id 
        LEFT JOIN
            `concept` m_value 
                ON measurement.value_as_concept_id = m_value.concept_id 
        LEFT JOIN
            `concept` m_unit 
                ON measurement.unit_concept_id = m_unit.concept_id 
        LEFT JOIn
            `visit_occurrence` v 
                ON measurement.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` m_visit 
                ON v.visit_concept_id = m_visit.concept_id 
        LEFT JOIN
            `concept` m_source_concept 
                ON measurement.measurement_source_concept_id = m_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurement_65837970_path <- file.path(
  intermediate_tables_dir,
  "measurement_65837970",
  "measurement_65837970_*.csv")
message(str_glue('The data will be written to {measurement_65837970_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_65837970_measurement_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  measurement_65837970_path,
  destination_format = "CSV")



## Format lab data

In [123]:
measurement_65837970_path <- file.path(
  intermediate_tables_dir,
  "measurement_65837970",
  "measurement_65837970_*.csv")

In [124]:
labs_df <- read_bq_export_from_workspace_bucket(measurement_65837970_path)

dim(labs_df)

head(labs_df, 5)

In [125]:
group_by(labs_df,measurement_concept_id,standard_concept_name) %>% summarize(n=n())

### Triglycerides

In [126]:
trig_df <- labs_df[labs_df$measurement_concept_id==3022192,]

In [127]:
#check concept names
group_by(trig_df,standard_concept_name) %>% summarize(n=n())

In [128]:
#check units
#this will require manual review, as there are a lot of inconsistent and inconsistently-named units which differ from trait to trait
group_by(trig_df,unit_source_value) %>% summarize(n=n())

In [129]:
#keep only consistent units
trig_df <- trig_df %>% filter(unit_source_value == 'mg/dL')

In [130]:
names(trig_df)

In [131]:
head(trig_df)

In [132]:
hist(trig_df$value_as_number)

In [133]:
#exclude non-physiologic values
#for triglycerides, this was defined generously as >1 and <1000 mg/dL
trig_df <- trig_df %>% filter(value_as_number>1 & value_as_number<1000)

In [134]:
hist(trig_df$value_as_number)

In [135]:
dim(trig_df)

In [136]:
dim(distinct(trig_df,person_id))

In [137]:
#keep just the earliest value for each subject
trig_df$measurement_datetime<-as.Date(trig_df$measurement_datetime)
trig_df <- trig_df %>% group_by(person_id) %>% filter(measurement_datetime == min(measurement_datetime))
dim(trig_df)

In [138]:
#some subjects still have multiple measurements. Keep just one reading per subject
trig_df <- distinct(trig_df,person_id,.keep_all=T)
dim(trig_df)
#view resulting distribution
hist(trig_df$value_as_number,breaks=100)

In [139]:
#keep "person_id," "value_as_number," and "measurement_datetime"
trig_df <- select(trig_df,person_id,value_as_number,measurement_datetime)
head(trig_df)

In [140]:
colnames(trig_df)<-c('person_id','Trig','Trig_measurement_datetime')
head(trig_df)

### LDL

In [141]:
#if undesired measurement types are present, filter
ldl_df <- labs_df[((labs_df$measurement_concept_id!=3022192) & 
                  (labs_df$measurement_concept_id!=3008631)
                    ),]
dim(ldl_df)

In [142]:
group_by(ldl_df,measurement_concept_id) %>% summarize(n=n())

In [143]:
group_by(ldl_df,standard_concept_name) %>% summarize(n=n())

In [144]:
#check units
ldl_df %>% group_by(unit_source_value) %>% summarize(n=n())

In [145]:
#keep only consistent units
ldl_df <- ldl_df %>% filter(unit_source_value == 'mg/dL' | unit_source_value == 'mg/dL{calc}')
dim(ldl_df)

In [146]:
hist(ldl_df$value_as_number)

In [147]:
#exclude non-physiologic values
ldl_df <- ldl_df %>% filter(value_as_number>1 & value_as_number<1000)

In [148]:
hist(ldl_df$value_as_number)

In [149]:
temp <- ldl_df %>% filter(value_as_number>1 & value_as_number<400)

In [150]:
hist(temp$value_as_number)

In [151]:
ldl_df <- temp

In [152]:
#keep just the earliest value for each subject
ldl_df$measurement_datetime<-as.Date(ldl_df$measurement_datetime)
ldl_df <- ldl_df %>% group_by(person_id) %>% filter(measurement_datetime == min(measurement_datetime))

In [153]:
dim(distinct(ldl_df,person_id))
dim(ldl_df)

In [154]:
#some subjects may still have multiple measurements. Keep just one reading per subject
ldl_df <- distinct(ldl_df,person_id,.keep_all=T)

#view resulting distribution
hist(ldl_df$value_as_number,breaks=100)

In [155]:
#keep "person_id," "value_as_number," and "measurement_datetime"
ldl_df <- select(ldl_df,person_id,value_as_number,measurement_datetime)
head(ldl_df)

In [156]:
colnames(ldl_df)<-c('person_id','LDL','LDL_measurement_datetime')
head(ldl_df)

### Troubleshooting

In [157]:
test <- left_join(ldl_df,trig_df,by='person_id')
dim(test)
head(test)

In [158]:
test %>% ggplot(aes(x=LDL, y=Trig)) + geom_point()

In [159]:
#mistake in original dataframe with LDL == trig
#get rid of all values where LDL=trig
#this may get rid of some people who have valid later values

In [160]:
ldl_trig <- full_join(ldl_df,trig_df,by='person_id')
dim(ldl_trig)
head(ldl_trig)

In [161]:
ldl_trig <- mutate(ldl_trig,diff = LDL-Trig)
head(ldl_trig)

In [162]:
hist(ldl_trig$diff,breaks=100)

In [ ]:
## This must have been an issue in V7. V8 does not seem to indicate the filtering below is warrented.

#get rid of those with ldl=trig since this seems to be a mistake

#ldl_trig <- ldl_trig[ldl_trig$diff!=0,]
#dim(ldl_trig)

In [ ]:
hist(ldl_trig$diff,breaks=100)

In [ ]:
ldl_trig %>% ggplot(aes(x=LDL, y=Trig)) + geom_point()

# Get demographics -DONE

## Do once (done)

In [163]:
# This query represents dataset "Temp ldl trig" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_09842086_person_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
            )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_09842086_path <- file.path(
  intermediate_tables_dir,
  "person_09842086",
  "person_09842086_*.csv")
message(str_glue('The data will be written to {person_09842086_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_09842086_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_09842086_path,
  destination_format = "CSV")

## Format demographic data

In [164]:
person_09842086_path <- file.path(
  intermediate_tables_dir,
  "person_09842086",
  "person_09842086_*.csv")
#person_09842086_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240321/person_09842086/person_09842086_*.csv'

In [165]:
demo_df <- read_bq_export_from_workspace_bucket(person_09842086_path)

dim(demo_df)

head(demo_df, 5)

In [166]:
#
demo_df %>% group_by(race) %>% summarize(n=n())

In [167]:
demo_df %>% group_by(sex_at_birth) %>% summarize(n=n())

In [168]:
demo_df$date_of_birth <- as.Date(demo_df$date_of_birth)
head(demo_df)

# Get BMI - DONE

## Do once - DONE

In [169]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "temp3" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v7
dataset_80780919_measurement_sql <- paste("
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (
                                    3038553
                                )       
                                AND full_text LIKE '%_rank1]%'      
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )  
                AND (
                    measurement.PERSON_ID IN (
                        SELECT
                            distinct person_id  
                        FROM
                            `cb_search_person` cb_search_person  
                        WHERE
                            cb_search_person.person_id IN (
                                SELECT
                                    person_id 
                                FROM
                                    `cb_search_person` p 
                                WHERE
                                    has_whole_genome_variant = 1 
                            ) 
                        )
                )
            ) measurement 
        LEFT JOIN
            `concept` m_standard_concept 
                ON measurement.measurement_concept_id = m_standard_concept.concept_id 
        LEFT JOIN
            `concept` m_type 
                ON measurement.measurement_type_concept_id = m_type.concept_id 
        LEFT JOIN
            `concept` m_operator 
                ON measurement.operator_concept_id = m_operator.concept_id 
        LEFT JOIN
            `concept` m_value 
                ON measurement.value_as_concept_id = m_value.concept_id 
        LEFT JOIN
            `concept` m_unit 
                ON measurement.unit_concept_id = m_unit.concept_id 
        LEFT JOIn
            `visit_occurrence` v 
                ON measurement.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` m_visit 
                ON v.visit_concept_id = m_visit.concept_id 
        LEFT JOIN
            `concept` m_source_concept 
                ON measurement.measurement_source_concept_id = m_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurement_80780919_path <- file.path(
  intermediate_tables_dir,
  "measurement_80780919",
  "measurement_80780919_*.csv")
message(str_glue('The data will be written to {measurement_80780919_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_80780919_measurement_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  measurement_80780919_path,
  destination_format = "CSV")

## Format BMI data

In [ ]:
#measurement_80780919_path <- 'gs://fc-secure-7e84f6f0-9e03-4626-b34e-6dcf5d5f1701/bq_exports/megan.lancaster@researchallofus.org/20240324/measurement_80780919/measurement_80780919_*.csv'

In [170]:
BMI_df <- read_bq_export_from_workspace_bucket(measurement_80780919_path)

dim(BMI_df)

head(BMI_df, 5)

In [171]:
group_by(BMI_df,measurement_concept_id,standard_concept_name) %>% summarize(n=n())

In [172]:
group_by(BMI_df,unit_source_value) %>% summarize(n=n())

In [173]:
BMI_df<-BMI_df %>% filter(unit_source_value == 'kg/m2')
dim(BMI_df)

In [174]:
#filter out non-physiologic and extreme values #BMI <14 or >60
BMI_df <- BMI_df %>% filter(value_as_number>14 & value_as_number<60)
dim(BMI_df)

In [175]:
hist(BMI_df$value_as_number)

In [176]:
#ok now keep just the ?earliest value for each person
BMI_df$measurement_datetime<-as.Date(BMI_df$measurement_datetime)
BMI_df <- BMI_df %>% group_by(person_id) %>% filter(measurement_datetime == min(measurement_datetime))
dim(BMI_df)

In [177]:
BMI_df <- distinct(BMI_df,person_id,.keep_all=T)
dim(BMI_df)
hist(BMI_df$value_as_number)

In [178]:
names(BMI_df)

In [179]:
BMI_df <- BMI_df %>% select('person_id','value_as_number','measurement_datetime')
head(BMI_df)
dim(BMI_df)

In [180]:
colnames(BMI_df) <- c('person_id','BMI','BMI_date')
head(BMI_df)

# Get censored ages - IN PROGRESS

In [224]:
DATASET <- Sys.getenv('WORKSPACE_CDR')
DATASET

In [225]:
download_data <- function(query) {
  tb <- bq_project_query(Sys.getenv('GOOGLE_PROJECT'), query = str_glue(query)
                         , default_dataset = Sys.getenv('WORKSPACE_CDR'))
  bq_table_download(tb)
}

In [226]:
query="
SELECT visit_concept_id, concept_name, COUNT(DISTINCT person_id) countp
FROM {DATASET}.visit_occurrence
JOIN {DATASET}.concept ON concept_id=visit_concept_id
GROUP BY 1,2
ORDER BY countp DESC
LIMIT 10
"
df = download_data(query)
dim(df)

In [227]:
df

In [228]:
query="
SELECT visit_concept_id, concept_name, visit_start_datetime, person_id
FROM {DATASET}.visit_occurrence
JOIN {DATASET}.concept ON concept_id=visit_concept_id
"
df = download_data(query)
dim(df)

In [229]:
head(df)

In [230]:
df %>% group_by(concept_name) %>% summarize(n=n())

In [231]:
#Select subset of visit types to use for censoring
#remove 'Mass Immunization Center', 'Laboratory Visit', 'Mammography Center', 'Unknown Value (but present in data)'
df2 <- df %>% filter(concept_name != 'Mass Immunization Center' & concept_name !='Laboratory Visit' &
                     concept_name != 'Mammography Center' & concept_name !='Unknown Value (but present in data)')

In [232]:
dim(df2)
head(df2)

In [233]:
df2$visit_start_datetime <- as.Date(df2$visit_start_datetime)

In [234]:
#icd_df <- icd_df %>% group_by(person_id) %>% filter(condition_start_datetime == min(condition_start_datetime))
#dim(icd_df)

In [235]:
#group_by person_id

#find last date before CAD diagnosis date
#codes_df
#if nothing earlier, use CAD date

#repeat for HC date

In [236]:
codes_df <- pheno_df2[,c('person_id','CAD_code','CAD_code_date')]

In [237]:
#dim(codes_df)
#head(codes_df)

In [238]:
nrow(distinct(codes_df,person_id))

In [239]:
df2 <- left_join(df2,codes_df, by='person_id')

In [240]:
dim(df2)
head(df2)

In [241]:
df2 <- df2 %>% arrange(person_id,visit_start_datetime)
head(df2)

In [242]:
#temp <- df2[1:10000,]
#temp

In [243]:
#nrow(distinct(temp,person_id))

In [244]:
temp <- df2

In [245]:
temp <- temp %>% group_by(person_id) %>% mutate(type = case_when(
             is.na(CAD_code_date) ~ 1,
             CAD_code_date <= min(visit_start_datetime) ~ 2,
             .default = 3
))
dim(temp)

In [246]:
nrow(distinct(temp,person_id))

In [247]:
temp %>% arrange(type)

In [248]:
dim(temp)
nrow(distinct(temp,person_id))

In [249]:
#type 3 CAD_code_date > min(visit_start_datetime)
#use filter to keep lines where type = 3 and visit_start_datetime < CAD_code_date
temp2 <- temp %>% filter(type == 1 | type == 2 | (type ==3 & visit_start_datetime < CAD_code_date))

In [250]:
dim(temp2)
nrow(distinct(temp2,person_id))

In [251]:
temp2 <- temp2 %>% group_by(person_id) %>% mutate(CAD_censored_date = case_when(
                    type == 1 ~ max(visit_start_datetime),
                    type == 2 ~ CAD_code_date,
                    type == 3 ~ max(visit_start_datetime) 
))

In [252]:
dim(temp2)
nrow(distinct(temp2,person_id))

In [253]:
nrow(distinct(temp2,person_id,CAD_censored_date))

In [254]:
head(temp2)

In [255]:
dim(distinct(temp2,person_id))

In [256]:
dim(temp2)

In [257]:
dim(distinct(temp2,person_id,CAD_censored_date))

In [258]:
temp3 <- distinct(temp2, person_id, CAD_censored_date)
dim(temp3)

In [259]:
#just keep: distinct person_id, CAD_censored_date, then join with pheno matrix

In [260]:
#save
write_excel_csv(
    temp3,
    file.path(
        intermediate_tables_dir_local,
        'CAD_censored_ages.csv'
    )
)

## Scratch

In [ ]:
#temp <- temp %>% filter(visit_start_datetime <= CAD_code_date | is.na(CAD_code_date))
#nrow(distinct(temp,person_id))

In [ ]:
#dim(temp)

In [ ]:
#temp <- temp %>% rowwise() %>% mutate(diff = difftime(CAD_code_date,visit_start_datetime,units="days"))
#head(temp)

In [ ]:
#temp <- temp %>%  arrange(person_id,diff)

In [ ]:
#temp2 <- temp %>% group_by(person_id) %>% filter()

#)

In [ ]:
#temp <- temp %>% group_by(person_id) %>% mutate(CAD_censored_date = case_when(
 #                                       is.na(CAD_code_date) ~ CAD_code_date,
  #                                      diff > 0 ~ 
#))

In [ ]:
temp <- #temp %>% group_by(person_id) %>% rowwise() %>% 
        #                                mutate(diff2 = case_when(
        #                                CAD_code_date>visit_start_datetime ~ difftime(CAD_code_date,visit_start_datetime,units="days"),
        #                                CAD_code_date<=visit_start_datetime~ as.difftime(0,units="days"))
)
temp

In [ ]:
#rrange(temp,visit_start_datetime)

In [ ]:
#diff = CAD_code_date - visit_start_datetime
#use minimum positive value

#temp <- temp %>% group_by(person_id) %>% mutate(
                                    #CAD_censored_date = case_when(
                                   # is.na(CAD_code_date) ~ max(visit_start_datetime),
                                    #.default = )
#temp
#f CAD_code_date:
#    CAD_censored_date = max(visit_start_datetime < CAD_code_date)
#else:
#    CAD_censored_date = max(visit_start_datetime)

# Join phenotype data

In [ ]:
#X demo: DoB, self-reported race, sex
#X LDL (earliest), triglycerides (earliest)
#CAD-related ICD, CPT codes
#X statin use 
#other cholesterol med use
#survey answera
#BMI
#dates for everything

In [181]:
pheno_df <- left_join(demo_df,ldl_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [182]:
pheno_df <- left_join(pheno_df,trig_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [183]:
pheno_df <- left_join(pheno_df,statin_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [184]:
#save intermediate file

write_excel_csv(
    pheno_df,
    file.path(
        intermediate_tables_dir_local,
        'LDLR_phenotype_intermediate_datasave.csv'
    )
)

In [185]:
pheno_df <- left_join(pheno_df,codes_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [186]:
pheno_df <- left_join(pheno_df,BMI_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [187]:
pheno_df <- left_join(pheno_df,meds_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [188]:
pheno_df %>% group_by(any_chol_med,on_statin) %>% summarize(n=n())

In [189]:
pheno_df$any_chol_med[pheno_df$any_chol_med=='no'] <- NA
pheno_df %>% group_by(any_chol_med,on_statin) %>% summarize(n=n())

In [190]:
#add columns for PAD, FH, and CAD FH

In [191]:
pheno_df <- left_join(pheno_df,PAD_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [192]:
pheno_df <- left_join(pheno_df,FH_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [193]:
pheno_df <- left_join(pheno_df,CADFH_df,by='person_id')
dim(pheno_df)
head(pheno_df)

In [194]:
#add hypercholesterolemia codes
pheno_df <- left_join(pheno_df,hc_df,by='person_id')
dim(pheno_df)
head(pheno_df)

# Format

In [195]:
names(pheno_df)

In [196]:
pheno_df %>% group_by(CADFH_code) %>% summarize(n=n())

In [197]:
#add column for if they were on a statin/cholesterol medication at the time of LDL measurement
pheno_df <- pheno_df %>% rowwise() %>% mutate(LDL_measured_on_meds = case_when(
                    LDL_measurement_datetime <= any_chol_med_start_date ~ 0,
                    LDL_measurement_datetime >  any_chol_med_start_date ~ 1))
pheno_df %>% group_by(LDL_measured_on_meds) %>% summarize(n=n())

In [198]:
pheno_df2 <- pheno_df[ ,c(
    'person_id','date_of_birth','race','sex_at_birth','LDL','LDL_measurement_datetime','Trig','Trig_measurement_datetime',
    'CAD_code','CAD_code_date','BMI','any_chol_med','any_chol_med_start_date','PAD_code','PAD_code_date',
    'FH_code','FH_code_date','CADFH_code','CADFH_code_date','HC_code_date','HC_code','LDL_measured_on_meds')]
names(pheno_df2)

In [199]:
colnames(pheno_df2) <- c('person_id','date_of_birth','race','sex_at_birth',
                         'LDL','Date_LDL_assessment',
                         'Trig','Date_Trig_assessment',
                         'CAD_code','CAD_code_date',
                         'BMI','any_chol_med','any_chol_med_start_date',
                        'PAD_code','PAD_code_date','FH_code','FH_code_date',
                        'CADFH_code','CADFH_code_date','HC_code_date','HC_code',
                         'LDL_measured_on_meds')
head(pheno_df2)

In [200]:
#pheno_df2 %>% group_by(CAD_code) %>% summarize(n=n())

In [201]:
#CAD_code, any_chol_med: 1 or 0 
#age at CAD dx
#pheno_df3 <- pheno_df2 %>% mutate(C)
pheno_df2$CAD_code[!is.na(pheno_df2$CAD_code)] <- 1
#pheno_df2 %>% group_by(CAD_code) %>% summarize(n=n())

In [202]:
pheno_df2$CAD_code[is.na(pheno_df2$CAD_code)] <- 0

In [203]:
pheno_df2 %>% group_by(any_chol_med) %>% summarize(n=n())

In [204]:
pheno_df2$any_chol_med[!is.na(pheno_df2$any_chol_med)] <- 1
pheno_df2$any_chol_med[is.na(pheno_df2$any_chol_med)] <- 0

In [205]:
pheno_df2 %>% group_by(any_chol_med) %>% summarize(n=n())

In [206]:
#age at CAD code
pheno_df2 <- pheno_df2 %>% mutate(CAD_age = year(CAD_code_date) - year(date_of_birth))
head(pheno_df2)

In [207]:
#age at LDL measurement
pheno_df2 <- pheno_df2 %>% mutate(Age_at_LDL_assessment = year(as.period(interval(date_of_birth,Date_LDL_assessment))))
head(pheno_df2)

In [208]:
#age at HC diagnostic code
pheno_df2 <- pheno_df2 %>% mutate(Age_at_HC = year(as.period(interval(date_of_birth,HC_code_date))))
head(pheno_df2)

In [209]:
#library(lubridate)

In [210]:
#pheno_df2 <- pheno_df2 %>% mutate(test = year(as.period(interval(date_of_birth,Date_LDL_assessment))))
#head(pheno_df2)

In [211]:
#hist(pheno_df2$diff,breaks=100)

In [212]:
#pheno_df2 %>% ggplot(aes(x=LDL, y=Trig)) + geom_point()

In [213]:
#pheno_df3 <- pheno_df2
#pheno_df3$LDL[pheno_df3$diff==0] = NA
#pheno_df3$Trig[pheno_df3$diff==0] = NA
#pheno_df3 %>% ggplot(aes(x=LDL, y=Trig)) + geom_point()

In [214]:
hist(pheno_df2$LDL,breaks=100)

In [215]:
hist(pheno_df$Trig,breaks=100)

In [216]:
names(pheno_df2)

In [217]:
pheno_df2 %>% group_by(CADFH_code) %>% summarize(n=n())

In [218]:
pheno_df2$CADFH_code[!is.na(pheno_df2$CADFH_code)] = TRUE
pheno_df2$CADFH_code[is.na(pheno_df2$CADFH_code)] = FALSE
pheno_df2 %>% group_by(CADFH_code) %>% summarize(n=n())

In [220]:
#save
write_excel_csv(
    pheno_df2,
    file.path(
        intermediate_tables_dir_local,
        'LDLR_phenotypes_formatted.csv'
    )
)

In [221]:
dim(pheno_df2)

In [222]:
dim(distinct(pheno_df2,person_id))

In [261]:
dim(temp3)

In [262]:
#join CAD-censored date
pheno_df3 <- left_join(pheno_df2, temp3, by='person_id')
head(pheno_df3)

In [263]:
dim(pheno_df3)
dim(distinct(pheno_df3,person_id))

In [264]:
#calculate CAD censored age
pheno_df3 <- pheno_df3 %>% mutate(CAD_Censored_Age = year(as.period(interval(date_of_birth,CAD_censored_date))))
head(pheno_df3)

In [265]:
write_excel_csv(
    pheno_df3,
    file.path(
        intermediate_tables_dir_local,
        'LDLR_phenotypes_formatted_censored_ages.csv'
    )
)